<a href="https://colab.research.google.com/github/Bruno-Paulo/PETs/blob/main/Distributed_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distributed Learning Tutorial

_Last run: May 1, 2025_


## 1. Introduction

This tutorial explores Distributed Learning using the [Flower](https://github.com/adap/flower) framework. You will learn how to train privacy-preserving machine learning models, ensuring sensitive data remains protected while still enabling valuable insights.



### 1.1 Scenario

LifeMed Analytics, a healthcare technology company, specialises in predictive analytics to improve patient care. Using machine learning, LifeMed helps healthcare providers identify patients at risk of stroke based on medical history, health indicators and lifestyle factors.

However, patient data is distributed across **five** hospitals and research facilities, making centralised model training impossible due to privacy regulations and data security concerns. LifeMed needed a way to collaboratively train models without sharing raw patient data.

### 1.2 Solution

Distributed learning allows LifeMed and its partner institutions to train a shared predictive model without sharing sensitive patient information. Instead of centralising data, each hospital trains a local model and only shares model updates. This protects patient privacy while improving model accuracy by leveraging diverse, real-world healthcare data.

### 1.3 Distributed Learning

Distributed Learning (DL) is a machine learning approach that enables collaborative model training across multiple data sources without requiring direct access to the underlying data. Rather than transferring raw data to a central location, DL ensures privacy by sharing only model updates, such as learned parameters or gradients.

A key method within distributed learning is **federated learning**, where individual devices or institutions train models locally and periodically send updates to a central aggregator. This approach allows diverse, decentralised datasets to contribute to model improvement while maintaining privacy. Other techniques such as **split learning** and **meta learning** offer alternative strategies to balance computational efficiency and security.  

While distributed learning enhances privacy and enables broader participation, it introduces challenges such as communication overhead, device heterogeneity, and potential privacy risks from model inversion attacks. Techniques such as differential privacy and secure aggregation help mitigate these risks, ensuring that DL remains a practical and scalable solution for real-world applications.



### 1.4 Outline of this tutorial

This tutorial will guide you through the implementation of distributed learning using the Flower framework. You will learn how to train machine learning models across multiple data sources while maintaining data privacy.

**Build a federated learning system**

- Define the machine learning model for stroke prediction.
- Implement a FlowerClient to handle local model training and evaluation.
- Set up a ServerApp to aggregate model updates using the FedAvg strategy.

**Run the simulation**

- Train the model in a centralised setup for comparison.
- Run the federated learning pipeline and analyse the results.
- Compare accuracy and loss between centralised and distributed models.

**Advanced customisation**

- Implement server-side parameter evaluation to ensure stable and consistent validation.
- Explore custom aggregation strategies.
- Learn how to send/receive custom parameters between clients and the server.



## 2. Setup

In order to apply distributed learning we will need to download the original data and then configure the Python framework that will assist in the tutorial.


---


***Note:*** This code will only work correctly if you use the Google
Chrome browser.



---





### 2.1 Downloading the original dataset

Please download the healthcare dataset in CSV file format from [here](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset).

Now we need attach the CSVs to this Google Colab notebook. Run the code below and then Click the "Choose Files" button. In the file picker choose the CSV file that you have just downloaded.


In [ ]:
from google.colab import files

# Optional: You can skip this step if you are running the code on your own
# machine
uploaded = files.upload()

Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data.csv


The csv files are now available in the folder `content/`.

### 2.2 Installing the Flower framework

Flower (flwr) is an open source framework for building federated AI systems, designed with **customisability, extensibility, and framework-agnostic support** in mind.  

It allows seamless integration with popular machine learning libraries such as PyTorch, TensorFlow and Hugging Face. With a focus on maintainability and community-driven development, Flower enables both research and production-scale federated learning applications.

We will use the Flower simulation engine to efficiently run a large number of clients without device management overhead.
It supports both federated learning and other distributed machine learning approaches through:

- **Client-Server Architecture:** Clients train models locally and send updates to a central server, which aggregates them to update the global model.

- **Customisable strategies:** Flower allows developers to implement custom aggregation and training strategies.

- **Scalability:** It supports training across a large number of clients, making it ideal for real-world applications.

If you need more help [Browse all tutorials](https://flower.ai/docs/framework/index.html) or visit the [documentation](https://flower.ai/docs/).

To install the Flower framework, run the following:




In [ ]:
!pip install -q flwr[simulation] flwr-datasets[vision] torch torchvision matplotlib scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.

### 2.3 Importing libraries

Now that we have all dependencies installed, we can import everything we need for this tutorial.

In [ ]:
from collections import OrderedDict
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets.utils.logging import disable_progress_bar
from torch.utils.data import DataLoader, Dataset, TensorDataset
from imblearn.over_sampling import SMOTE

import torch.optim as optim
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from flwr_datasets.partitioner import IidPartitioner

import flwr
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import Metrics, Context, ndarrays_to_parameters, NDArrays, Scalar
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg, FedAdagrad, Strategy
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset

DEVICE = torch.device("cpu")  # Try "cuda" to train on GPU
print(f"Training on {DEVICE}")
print(f"Flower {flwr.__version__} / PyTorch {torch.__version__}")
disable_progress_bar()

Training on cpu
Flower 1.17.0 / PyTorch 2.6.0+cu124




---
***Note***

It is possible to switch to a runtime that has GPU acceleration enabled (on Google Colab: `Runtime > Change runtime type > Hardware accelerator: GPU > Save`). However, that Google Colab is not always able to offer GPU acceleration. If you see an error related to GPU availability in one of the following sections, consider switching back to CPU-based execution by setting `DEVICE = torch.device("cpu")`. If the runtime has GPU acceleration enabled, you should see the output `Training on cuda`, otherwise it'll say `Training on cpu`.



---



## 3. Operations on Data

### 3.1 Pre-process the data

We will create the load_data function which will load and pre-process the data. This process is divided into smaller steps and functions.




#### 3.1.1 Handle missing values and drop unnecessary columns

We clean the dataset by handling missing values and removing irrelevant columns. We also shuffle the dataset before splitting it into smaller datasets to simulate the distributed environment.

In [ ]:
def clean_data(df):
    df = df.drop(columns=["id"])  # Drop irrelevant columns
    df["bmi"] = df["bmi"].fillna(df["bmi"].median())  # Fill missing BMI values

    # Shuffle the dataset before partitioning
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

#### 3.1.2 Split features and target

We separate the input features (X) from the target variable (y).

In [ ]:
def split_features_target(df):
    X = df.drop(columns=["stroke"])
    y = df["stroke"]
    return X, y

#### 3.1.3 Define categorical and numerical features

We define transformations for numerical and categorical columns.

In [ ]:
def get_feature_transformers(X):
    """Create preprocessing pipelines for numerical and categorical features."""
    categorical_features = X.select_dtypes(include=["object"]).columns
    numeric_features = X.select_dtypes(include=["float64", "int64"]).columns

    numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
    categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    return preprocessor

#### 3.1.4 Preprocess categorical and numerical features

We apply the previously defined transformations.

In [ ]:
def preprocess_features(X):
    """Apply transformations to numerical and categorical features."""
    preprocessor = get_feature_transformers(X)
    X_transformed = preprocessor.fit_transform(X)
    return pd.DataFrame(X_transformed)  # Convert back to DataFrame for compatibility with SMOTE

#### 3.1.5 Data partitioning

We split the dataset into smaller datasets to simulate the distributed environment.

In [ ]:
def partition_data(X, y, num_partitions, partition_id):
    """Split data into partitions before applying SMOTE."""
    partitions = np.array_split(X, num_partitions)
    partition_data = partitions[partition_id]  # Select the relevant partition

    # Ensure corresponding labels are selected
    partition_indices = partition_data.index if hasattr(partition_data, 'index') else np.arange(len(partition_data))
    y_partition = y.iloc[partition_indices]

    return partition_data, y_partition

#### 3.1.6 Handle class imbalance with SMOTE

SMOTE is used to balance the dataset.

In [ ]:
def balance_classes(X, y):
    return SMOTE().fit_resample(X, y)

#### 3.1.7 Split data into train, validation, and test sets

We split the dataset into training, validation, and testing sets.

In [ ]:
def split_data(X, y, test_size=0.3, val_size=0.5, random_state=42):
    """Split the dataset into train, validation, and test sets."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=val_size, random_state=random_state)
    return X_train, X_val, X_test, y_train, y_val, y_test

#### 3.1.8 Convert data to pyTorch datasets

We define a custom dataset class and create DataLoaders.

In [ ]:
# Custom Dataset class
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def create_dataloaders(X_train, X_val, X_test, y_train, y_val, y_test):
    """Convert data into PyTorch datasets and create DataLoaders."""
    train_dataset = CustomDataset(X_train.values, y_train)
    val_dataset = CustomDataset(X_val.values, y_val)
    test_dataset = CustomDataset(X_test.values, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    return train_loader, val_loader, test_loader

#### 3.1.9 Load and process data using these functions

First we define the number of partitions we want to use in our distributed environment.
Then, we call all the above functions in sequence to load and process the data.

In [ ]:
NUM_PARTITIONS = 5 # Number of providers

In [ ]:
def load_data(partition_id: int, num_partitions: int = NUM_PARTITIONS):
    """Load, preprocess, balance, and split data, then return DataLoaders."""
    df = pd.read_csv("/content/healthcare-dataset-stroke-data.csv")
    df = clean_data(df)
    X, y = split_features_target(df)
    X = preprocess_features(X)
    X_partitioned, y_partitioned = partition_data(X, y, num_partitions, partition_id)
    X_resampled, y_resampled = balance_classes(X_partitioned, y_partitioned)
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_resampled, y_resampled)
    return create_dataloaders(X_train, X_val, X_test, y_train, y_val, y_test)

### 3.2 Defining the model

We define our **StrokePredictionModel** along with training and testing functions to evaluate how well a trained model performs in predicting strokes. The aim is to compare the performance of a centralised model trained with a distributed model trained to assess the impact of using distributed learning for machine learning tasks.

#### 3.2.1 Stroke prediction model class

In [ ]:
class StrokePredictionModel(nn.Module):
    def __init__(self, input_dim: int):
        super(StrokePredictionModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.output = nn.Linear(64, 1)  # Single output for binary classification

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.output(x)
        return x  # Raw logits, suitable for BCEWithLogitsLoss

#### 3.2.2 Train function

In [ ]:
def train(model, train_loader, epochs: int, verbose=False):
    """Train the model on the training set."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=1)
    model.to(DEVICE)
    model.train()

    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE).float()
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()  # Squeeze to match target shape
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            # Metrics
            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities > 0.5).float()
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader.dataset)
        epoch_acc = correct / total
        if verbose:
            print(f"Epoch {epoch+1}: Train Loss {epoch_loss:.4f}, Accuracy {epoch_acc:.4f}")

#### 3.2.3 Test function

In [ ]:
def test(model, test_loader):
    """Evaluate the model on the test set."""
    criterion = nn.BCEWithLogitsLoss()
    loss = 0.0
    correct = 0
    total = 0
    model.to(DEVICE)
    model.eval()

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE).float()
            outputs = model(X_batch).squeeze()
            loss += criterion(outputs, y_batch).item()

            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities > 0.5).float()
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    loss /= len(test_loader.dataset)
    accuracy = correct / total
    #print(f"Test Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")
    return loss, accuracy

### 3.3 Federated Learning with Flower

In the centralised training pipeline, all data is combined in one place for training. However, centralised training isn't feasible in scenarios where data cannot be combined for privacy or logistical reasons like in our use case scenario.

Federated learning addresses this by allowing decentralised training on separate datasets across multiple providers.

This tutorial transforms our centralised code into a federated learning pipeline using Flower and PyTorch.

Federated learning systems consist of a server and multiple clients. In Flower, we create a `ClientApp` and a `ServerApp` to run the client-side and server-side code, respectively.

The server is responsible for sending the global model's parameters to the clients, while the clients train locally using these parameters and their own data and send back updated parameters (or gradients) to the server.

#### 3.3.1 Update model parameters

We need two helper functions to update the local model with parameters received from the server and to get the updated model parameters from the local model:

- **`set_parameters`:** Updates the local model with parameters from the server.

- **`get_parameters`:** Retrieves the updated local model parameters to send back to the server.

In [ ]:
def set_parameters(net, parameters: List[np.ndarray]):
    """Set model parameters from a list of NumPy arrays."""
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict()
    for key, param in params_dict:
        # Ensure the parameter is converted to the correct shape and type
        state_dict[key] = torch.tensor(param, dtype=net.state_dict()[key].dtype)
    net.load_state_dict(state_dict, strict=True)


def get_parameters(net) -> List[np.ndarray]:
    return [val.cpu().numpy() for _, val in net.state_dict().items()]



---

***Note***: These functions use PyTorch’s `state_dict` to access model parameters, converting them to/from NumPy arrays for easy serialization by Flower.



---



#### 3.3.2 Define the Flower ClientApp



##### 3.3.2.1 ClientApp

The first step towards creating a `ClientApp` is to implement a subclasses of `flwr.client.Client` or `flwr.client.NumPyClient`. We use `NumPyClient` in this tutorial because it is easier to implement.

To do so, we create a subclass that implements the three methods `get_parameters`, `fit`, and `evaluate`:

* `get_parameters`: share current model parameters with the server.
* `fit`: receive global model parameters, train locally, and return updated parameters.
* `evaluate`: evaluate the global model on local data and return metrics.

We also pass the `partition_id` to the client and use it to log additional details.

Our clients will use the previously defined PyTorch components for model training and evaluation. Let's see a simple Flower client implementation that brings everything together:

In [ ]:
class FlowerClient(NumPyClient):
    def __init__(self, partition_id, net, trainloader, valloader):
        self.partition_id = partition_id
        self.net = net
        self.trainloader = trainloader
        self.valloader = valloader

    def get_parameters(self, config):
        print(f"[Client {self.partition_id}] get_parameters")
        return get_parameters(self.net)

    def fit(self, parameters, config):
        print(f"[Client {self.partition_id}] fit, config: {config}")
        set_parameters(self.net, parameters)
        train(self.net, self.trainloader, epochs=1)
        return get_parameters(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        print(f"[Client {self.partition_id}] evaluate, config: {config}")
        set_parameters(self.net, parameters)
        loss, accuracy = test(self.net, self.valloader)
        return float(loss), len(self.valloader), {"accuracy": float(accuracy)}

Each instance of `FlowerClient` represents a *single client* in a federated learning system. For example, if there are 5 clients, there will be 5 `FlowerClient` instances with each handling its own data and participating in training or evaluation when selected.

##### 3.3.2.2 The `client_fn` callback

In this tutorial we will simulate five clients on a single machine. To optimise memory usage, Flower creates `FlowerClient` instances only when needed, using a helper function called `client_fn`. This function dynamically creates a `FlowerClient` for a particular client, identified by a `partition-id`, ensuring that each client runs on its own data partition.  

The `client_fn` callback allows Flower to efficiently manage resources by:  

- Loading the associated data partition (`partition-id`) for a client.  
- Creating and returning a `FlowerClient` instance only when needed.  

This dynamic approach improves memory efficiency, making it ideal for multi-client simulations.

In [ ]:
def client_fn(context: Context) -> Client:
    """Create a Flower client representing a single organization."""
    # Load data
    # Note: each client gets a different trainloader/valloader, so each client
    # will train and evaluate on their own unique data partition
    # Read the node_config to fetch data partition associated to this node
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]

    trainloader, valloader, _ = load_data(partition_id, num_partitions)

    # Load model
    net = StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]).to(DEVICE)

    # Create a single Flower client representing a single organization
    # FlowerClient is a subclass of NumPyClient, so we need to call .to_client()
    # to convert it to a subclass of `flwr.client.Client`
    return FlowerClient(partition_id, net, trainloader, valloader).to_client()

# Create the ClientApp
client = ClientApp(client_fn=client_fn)

With this, we have the class `FlowerClient` which defines client-side training/evaluation and `client_fn` which allows Flower to create `FlowerClient` instances whenever it needs to call `fit` or `evaluate` on one particular client.
Also, we created an instance of `ClientApp` and passed it the `client_fn`.

`ClientApp` is the entrypoint that a running Flower client uses to call your code.

#### 3.3.3 Define the Flower ServerApp



##### 3.3.3.1 Choosing a strategy

A strategy sits at the core of the Federated Learning experiment. It defines how the server manages FL rounds, including:

- Sampling clients for training.
- Sending the global model to clients.
- Aggregating updated models received from clients.
- Performing evaluations.

Flower comes with many [strategies built-in](https://github.com/adap/flower/tree/main/src/py/flwr/server/strategy).

For this tutorial we will use the popular `FedAvg` strategy, which averages client models to create a new global model. This simple approach works well in many scenarios. See the [FedAvg paper](https://arxiv.org/abs/1602.05629) for more details.

In [ ]:
# Create FedAvg strategy
strategy = FedAvg(
    fraction_fit=1.0,  # Sample 100% of available clients for training
    fraction_evaluate=0.5,  # Sample 50% of available clients for evaluation
    min_fit_clients=5,  # Never sample less than 5 clients for training
    min_evaluate_clients=3,  # Never sample less than 3 clients for evaluation
    min_available_clients=5,  # Wait until all 5 clients are available
)

##### 3.3.3.2 The `server_fn` callback

Similar to `ClientApp`, we create a `ServerApp` using a utility function `server_fn`. It:

- Configures the number of FL rounds (`num_rounds`).
- Defines the strategy to use (in this case `FedAvg`).

The function returns a `ServerAppComponents` object, which Flower uses to manage server-side operations.

In [ ]:
def server_fn(context: Context) -> ServerAppComponents:
    """Construct components that set the ServerApp behaviour.
    You can use the settings in `context.run_config` to parameterize the
    construction of all elements (e.g the strategy or the number of rounds)
    wrapped in the returned ServerAppComponents object.
    """
    # Configure the server for 20 rounds of training
    config = ServerConfig(num_rounds=20)
    return ServerAppComponents(strategy=strategy, config=config)

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

#### 3.3.4 Launching the Simulation

In simulation, we often want to control the amount of resources each client can use. In the next cell, we specify a `backend_config` dictionary with the `client_resources` key (required) for defining the amount of CPU and GPU resources each client can access.

In [ ]:
# Specify the resources each of your clients need
# By default, each client will be allocated 1x CPU and 0x GPUs
backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 0.0}}

# When running on GPU, assign an entire GPU for each client
if DEVICE.type == "cuda":
    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 1.0}}
    # Refer to our Flower framework documentation for more details about Flower simulations
    # and how to set up the `backend_config`

The last step is the actual call to `run_simulation`. The function accepts a number of arguments:
- `server_app` and `client_app`: Server and client configurations.
- `num_supernodes`: Number of clients to simulate.
- `backend_config`: Client resource allocation.

In [ ]:
# Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)

DEBUG:flwr:Asyncio event loop already running.
INFO :      Starting Flower ServerApp, config: num_rounds=20, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
(pid=1453) 2025-04-11 09:00:50.416203: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=1453) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=1453) E0000 00:00:1744362050.790357    1453 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=1453) E0000 00:00:1744362050.829253    1453 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO :      Received initial parameters from one random client
INFO :      Starting

(ClientAppActor pid=1454) [Client 3] get_parameters
(ClientAppActor pid=1454) [Client 0] fit, config: {}


(ClientAppActor pid=1454) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=1454)   return bound(*args, **kwds)
(ClientAppActor pid=1454) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=1454)   return bound(*args, **kwds)


(ClientAppActor pid=1454) [Client 2] fit, config: {}


(ClientAppActor pid=1454) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=1454)   return bound(*args, **kwds)


...

(ClientAppActor pid=1453) [Client 3] evaluate, config: {} [repeated 12x across cluster]


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 18]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)


(ClientAppActor pid=1453) [Client 4] fit, config: {} [repeated 20x across cluster]


INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 19]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 20]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)
(ClientAppActor pid=1453) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead. [repeated 29x across cluster]
(ClientAppAct

(ClientAppActor pid=1453) [Client 3] evaluate, config: {} [repeated 12x across cluster]
(ClientAppActor pid=1453) [Client 4] fit, config: {} [repeated 15x across cluster]


<!--So how does this work? How does Flower execute this simulation?

When we call `run_simulation`, we tell Flower that there are 5 clients (`num_supernodes=5`, where 1 `SuperNode` launches 1 `ClientApp`). Flower then goes ahead an asks the `ServerApp` to issue an instructions to those nodes using the `FedAvg` strategy. `FedAvg` knows that it should select 100% of the available clients (`fraction_fit=1.0`), so it goes ahead and selects 5 random clients (i.e., 100% of 5).

Flower then asks the selected 5 clients to train the model. Each of the 5 `ClientApp` instances receives a message, which causes it to call `client_fn` to create an instance of `FlowerClient`. It then calls `.fit()` on each the `FlowerClient` instances and returns the resulting model parameter updates to the `ServerApp`. When the `ServerApp` receives the model parameter updates from the clients, it hands those updates over to the strategy (*FedAvg*) for aggregation. The strategy aggregates those updates and returns the new global model, which then gets used in the next round of federated learning.
-->

Here's what happens during the simulation:

1. Flower creates `num_supernodes` clients (e.g. 5 for this tutorial).
2. Server selects clients for training based on strategy (e.g. `FedAvg` selects all 5 clients).
3. Clients train locally and send updated models back to the server.
4. The server aggregates the updates into a new global model and repeats this for the specified number of rounds.

#### 3.3.5 Accuracy

Metrics such as loss are automatically aggregated by Flower. However, custom metrics such as **accuracy** require additional handling.

Flower doesn't know how to combine custom metrics, so we need to define an aggregation function and tell Flower to use it.

In this tutorial, we'll create a `weighted_average` function to aggregate the `accuracy` values returned by clients during evaluation. We'll pass this function to the `evaluate_metrics_aggregation_fn` callback of the strategy.

With this setup, the server will calculate a single `accuracy` metric based on the client scores after each round.


##### 3.3.5.1 Weighted average function

In [ ]:
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metric (weighted average)
    return {"accuracy": sum(accuracies) / sum(examples)}

##### 3.3.5.2 Updated `server_fn` callback

In [ ]:
def server_fn(context: Context) -> ServerAppComponents:
    """Construct components that set the ServerApp behaviour.
    You can use settings in `context.run_config` to parameterize the
    construction of all elements (e.g the strategy or the number of rounds)
    wrapped in the returned ServerAppComponents object.
    """

    # Create FedAvg strategy
    strategy = FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=0.5,
        min_fit_clients=5,
        min_evaluate_clients=3,
        min_available_clients=5,
        evaluate_metrics_aggregation_fn=weighted_average,  # <-- pass the metric aggregation function
    )

    # Configure the server for 20 rounds of training
    config = ServerConfig(num_rounds=20)
    return ServerAppComponents(strategy=strategy, config=config)

# Create a new server instance with the updated FedAvg strategy
server = ServerApp(server_fn=server_fn)

We now have a full system that performs federated training and federated evaluation. It uses the `weighted_average` function to aggregate custom evaluation metrics and calculates a single `accuracy` metric across all clients on the server side.

The other two categories of metrics (`losses_centralized` and `metrics_centralized`) are still empty because they only apply when centralized evaluation is being used.

With this setup, our federated learning system is complete. It trains a global model collaboratively across clients and evaluates it using custom metrics aggregated on the server.

Now, let's train both the centralised and distributed models and compare their results.

### 3.4 Training the model

Finally, we can start training!

We will benchmark the centralised model against a distributed model and see the differences.

#### 3.4.1 Train the centralised model

This simulates the reality of most of today's machine learning projects: each organisation has its own data, and the models are trained only on this internal data.

In [ ]:
# Training loop for each insurance provider
for partition_id in range(NUM_PARTITIONS):
    # Load data for the current partition
    trainloader, valloader, testloader = load_data(partition_id, NUM_PARTITIONS)

    # Initialize the model
    net = StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]).to(DEVICE)

    # Train the model
    for epoch in range(20): #Number of EPOCHS on each partition
        train(net, trainloader, 1)
        val_loss, val_accuracy = test(net, valloader)
        print(f"Epoch {epoch + 1}: validation loss {val_loss:.4f}, accuracy {val_accuracy:.4f}")

    # Evaluate on the test set
    test_loss, test_accuracy = test(net, testloader)
    print(f"Test set performance:\n\tLoss: {test_loss:.4f}\n\tAccuracy: {test_accuracy:.4f}")

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Epoch 1: validation loss 0.0126, accuracy 0.8299
Epoch 2: validation loss 0.0116, accuracy 0.8472
Epoch 3: validation loss 0.0113, accuracy 0.8507
Epoch 4: validation loss 0.0111, accuracy 0.8542
Epoch 5: validation loss 0.0107, accuracy 0.8438
Epoch 6: validation loss 0.0111, accuracy 0.8472
Epoch 7: validation loss 0.0105, accuracy 0.8646
Epoch 8: validation loss 0.0106, accuracy 0.8542
Epoch 9: validation loss 0.0109, accuracy 0.8715
Epoch 10: validation loss 0.0103, accuracy 0.8403
Epoch 11: validation loss 0.0100, accuracy 0.8611
Epoch 12: validation loss 0.0102, accuracy 0.8681
Epoch 13: validation loss 0.0096, accuracy 0.8611
Epoch 14: validation loss 0.0098, accuracy 0.8681
Epoch 15: validation loss 0.0097, accuracy 0.8785
Epoch 16: validation loss 0.0102, accuracy 0.8750
Epoch 17: validation loss 0.0097, accuracy 0.8715
Epoch 18: validation loss 0.0096, accuracy 0.8646
Epoch 19: validation loss 0.0094, accuracy 0.8646
Epoch 20: validation loss 0.0089, accuracy 0.8889
Test set 

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Epoch 1: validation loss 0.0204, accuracy 0.7801
Epoch 2: validation loss 0.0235, accuracy 0.5326
Epoch 3: validation loss 0.0210, accuracy 0.6838
Epoch 4: validation loss 0.0185, accuracy 0.7629
Epoch 5: validation loss 0.0127, accuracy 0.8247
Epoch 6: validation loss 0.0109, accuracy 0.8660
Epoch 7: validation loss 0.0117, accuracy 0.8557
Epoch 8: validation loss 0.0122, accuracy 0.8419
Epoch 9: validation loss 0.0122, accuracy 0.8591
Epoch 10: validation loss 0.0141, accuracy 0.8213
Epoch 11: validation loss 0.0103, accuracy 0.8832
Epoch 12: validation loss 0.0098, accuracy 0.8935
Epoch 13: validation loss 0.0101, accuracy 0.8694
Epoch 14: validation loss 0.0108, accuracy 0.8763
Epoch 15: validation loss 0.0101, accuracy 0.8763
Epoch 16: validation loss 0.0093, accuracy 0.8900
Epoch 17: validation loss 0.0102, accuracy 0.8694
Epoch 18: validation loss 0.0104, accuracy 0.8797
Epoch 19: validation loss 0.0090, accuracy 0.9003
Epoch 20: validation loss 0.0077, accuracy 0.9072
Test set 

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Epoch 1: validation loss 0.0121, accuracy 0.8542
Epoch 2: validation loss 0.0121, accuracy 0.8678
Epoch 3: validation loss 0.0119, accuracy 0.8542
Epoch 4: validation loss 0.0094, accuracy 0.8847
Epoch 5: validation loss 0.0104, accuracy 0.8847
Epoch 6: validation loss 0.0073, accuracy 0.9186
Epoch 7: validation loss 0.0080, accuracy 0.9186
Epoch 8: validation loss 0.0080, accuracy 0.9186
Epoch 9: validation loss 0.0072, accuracy 0.9322
Epoch 10: validation loss 0.0077, accuracy 0.9322
Epoch 11: validation loss 0.0068, accuracy 0.9424
Epoch 12: validation loss 0.0062, accuracy 0.9424
Epoch 13: validation loss 0.0059, accuracy 0.9390
Epoch 14: validation loss 0.0061, accuracy 0.9492
Epoch 15: validation loss 0.0066, accuracy 0.9322
Epoch 16: validation loss 0.0058, accuracy 0.9356
Epoch 17: validation loss 0.0051, accuracy 0.9525
Epoch 18: validation loss 0.0055, accuracy 0.9424
Epoch 19: validation loss 0.0056, accuracy 0.9525
Epoch 20: validation loss 0.0042, accuracy 0.9492
Test set 

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Epoch 1: validation loss 0.0168, accuracy 0.7354
Epoch 2: validation loss 0.0106, accuracy 0.8694
Epoch 3: validation loss 0.0098, accuracy 0.8866
Epoch 4: validation loss 0.0102, accuracy 0.9107
Epoch 5: validation loss 0.0102, accuracy 0.8591
Epoch 6: validation loss 0.0097, accuracy 0.8625
Epoch 7: validation loss 0.0090, accuracy 0.9038
Epoch 8: validation loss 0.0086, accuracy 0.8866
Epoch 9: validation loss 0.0100, accuracy 0.8832
Epoch 10: validation loss 0.0139, accuracy 0.8282
Epoch 11: validation loss 0.0084, accuracy 0.9278
Epoch 12: validation loss 0.0094, accuracy 0.9347
Epoch 13: validation loss 0.0082, accuracy 0.9278
Epoch 14: validation loss 0.0073, accuracy 0.9210
Epoch 15: validation loss 0.0077, accuracy 0.9107
Epoch 16: validation loss 0.0075, accuracy 0.9416
Epoch 17: validation loss 0.0067, accuracy 0.9347
Epoch 18: validation loss 0.0066, accuracy 0.9347
Epoch 19: validation loss 0.0067, accuracy 0.9313
Epoch 20: validation loss 0.0060, accuracy 0.9244
Test set 

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Epoch 1: validation loss 0.0173, accuracy 0.7041
Epoch 2: validation loss 0.0129, accuracy 0.7925
Epoch 3: validation loss 0.0126, accuracy 0.8095
Epoch 4: validation loss 0.0117, accuracy 0.8401
Epoch 5: validation loss 0.0111, accuracy 0.8367
Epoch 6: validation loss 0.0109, accuracy 0.8333
Epoch 7: validation loss 0.0107, accuracy 0.8571
Epoch 8: validation loss 0.0094, accuracy 0.8776
Epoch 9: validation loss 0.0097, accuracy 0.8776
Epoch 10: validation loss 0.0103, accuracy 0.8571
Epoch 11: validation loss 0.0089, accuracy 0.8776
Epoch 12: validation loss 0.0095, accuracy 0.8605
Epoch 13: validation loss 0.0089, accuracy 0.8776
Epoch 14: validation loss 0.0095, accuracy 0.8946
Epoch 15: validation loss 0.0087, accuracy 0.8912
Epoch 16: validation loss 0.0077, accuracy 0.9184
Epoch 17: validation loss 0.0086, accuracy 0.9048
Epoch 18: validation loss 0.0076, accuracy 0.9082
Epoch 19: validation loss 0.0078, accuracy 0.9150
Epoch 20: validation loss 0.0079, accuracy 0.8980
Test set 

Training the simple StrokePredictionModel on our healthcare dataset split for 20 epochs should result in a test set accuracy of about 65%-90%, with a loss lower than 0.01.

#### 3.4.2 Train the distributed model

Let's train our distributed model. We do this by calling the `run_simulation` function with the parameters previously defined in the above sections plus the accuracy function.


In [ ]:
# Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)

INFO :      Starting Flower ServerApp, config: num_rounds=20, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
(pid=2239) 2025-04-11 09:02:08.689695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=2239) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=2239) E0000 00:00:1744362128.772464    2239 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=2239) E0000 00:00:1744362128.801074    2239 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(ClientAppActor pid=2240) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxe

(ClientAppActor pid=2240) [Client 1] get_parameters


(ClientAppActor pid=2240) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2240)   return bound(*args, **kwds)


(ClientAppActor pid=2240) [Client 0] fit, config: {}


(ClientAppActor pid=2240) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2240)   return bound(*args, **kwds)


...

INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 19]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)


(ClientAppActor pid=2239) [Client 4] evaluate, config: {} [repeated 9x across cluster]


INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 20]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 20 round(s) in 36.03s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.017416392584722756
INFO :      		round 2: 0.01444803186359706
INFO :      		round 3: 0.013345072439280422
INFO :      		round 4: 0.013548522758894985
INFO :      		round 5: 0.014305254991596567
INFO :      		round 6: 0.012994027960392078
INFO :      		round 7: 0.01452319047237443
INFO :      		round 8: 0.01350792894739958
INFO :      		round 9: 0.0132983

(ClientAppActor pid=2240) [Client 4] fit, config: {} [repeated 11x across cluster]
(ClientAppActor pid=2239) [Client 4] evaluate, config: {} [repeated 6x across cluster]


(ClientAppActor pid=2239) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead. [repeated 20x across cluster]
(ClientAppActor pid=2239)   return bound(*args, **kwds) [repeated 20x across cluster]


We achieved an accuracy of 77%-83% with a loss around 0.13.
Let's now compare the results of both models.

#### 3.4.3 Comparing the results

The **centralised model**, trained on the full dataset for 20 epochs, achieves a test accuracy between **65% and 90%**. This approach benefits from direct access to all data, but requires the sharing of raw data, which raises privacy concerns.  

In contrast, the **distributed model** trained over 20 rounds using federated learning achieves a final accuracy of **~83%**, with a steady improvement over the rounds. This approach preserves privacy while achieving comparable performance to centralised training. Although federated training introduces communication overhead, it allows learning from decentralised data sources without direct data exchange.

### 3.5 Advanced Features

The sections below cover advanced features and customizations for federated learning using Flower. These are optional but can enhance your system based on specific needs.

#### 3.5.1 Strategy customisation


##### 3.5.1.1 Server-side parameter **initialisation**

By default, Flower initialises the global model by asking a random client for its initial parameters. However, in many cases we may want more control over the initial parameters. Flower allows us to set these initial parameters directly by passing them to the strategy. Here's how:

1. Create the initial model (e.g. StrokePredictionModel()) and retrieve its parameters.
2. Pass these parameters to the `FedAvg` strategy using the `initial_parameters` argument.
3. Run the simulation with the customised strategy, and Flower will skip asking each client for initial parameters.

<!--
Next, we create a `server_fn` that returns the components needed for the server. Within `server_fn`, we create a Strategy that uses the initial parameters.

Passing `initial_parameters` to the `FedAvg` strategy prevents Flower from asking one of the clients for the initial parameters. In `server_fn`, we pass this new `strategy` and a `ServerConfig` for defining the number of federated learning rounds (`num_rounds`).

Similar to the `ClientApp`, we now create the `ServerApp` using the `server_fn` and specify the resources for each client and run the simulation.:

-->



###### 3.5.1.1.1 Define the model

In [ ]:
# 1. Create an instance of the model and get the parameters
params = get_parameters(StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]))

###### 3.5.1.1.2 Pass the parameters to the strategy

In [ ]:
# 2. Pass these parameters to the FedAvg strategy using the initial_parameters argument.
def server_fn(context: Context) -> ServerAppComponents:
    # Create FedAvg strategy
    strategy = FedAvg(
        fraction_fit=0.3,
        fraction_evaluate=0.3,
        min_fit_clients=3,
        min_evaluate_clients=3,
        min_available_clients=NUM_PARTITIONS,
        initial_parameters=ndarrays_to_parameters(
            params
        ),  # Pass initial model parameters
    )

    # Configure the server for 3 rounds of training
    config = ServerConfig(num_rounds=3)
    return ServerAppComponents(strategy=strategy, config=config)

###### 3.5.1.1.3 Run the simulation with the customised strategy

In [ ]:
# 3. Run the simulation with the customised strategy
# Create ServerApp
server = ServerApp(server_fn=server_fn)

# Specify the resources each of your clients need
# If set to none, by default, each client will be allocated 2x CPU and 0x GPUs
backend_config = {"client_resources": None}
if DEVICE.type == "cuda":
    backend_config = {"client_resources": {"num_gpus": 1}}

# Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)

INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 5)
(pid=2922) 2025-04-11 09:03:08.568195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=2922) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=2922) E0000 00:00:1744362188.609409    2922 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=2922) E0000 00:00:1744362188.621709    2922 cuda_blas.cc:1418] Unable to register cuBLAS factory: Atte

(ClientAppActor pid=2922) [Client 1] fit, config: {}


(ClientAppActor pid=2922) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2922)   return bound(*args, **kwds)


(ClientAppActor pid=2922) [Client 3] fit, config: {}


(ClientAppActor pid=2922) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2922)   return bound(*args, **kwds)
INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)


...

(ClientAppActor pid=2922) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2922)   return bound(*args, **kwds)
INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)


(ClientAppActor pid=2922) [Client 4] fit, config: {}


(ClientAppActor pid=2922) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2922)   return bound(*args, **kwds)
(ClientAppActor pid=2922) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2922)   return bound(*args, **kwds)
(ClientAppActor pid=2922) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=2922)   return bound(*args, **kwds)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      


(ClientAppActor pid=2922) [Client 1] evaluate, config: {}
(ClientAppActor pid=2922) [Client 2] evaluate, config: {}
(ClientAppActor pid=2922) [Client 4] evaluate, config: {}


INFO :      [SUMMARY]
INFO :      Run finished 3 round(s) in 23.74s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.01595286018326606
INFO :      		round 2: 0.014725497003995756
INFO :      		round 3: 0.0146282261546885
INFO :      




---
***Note***

This ensures full control over how the global model starts training.

If we look closely, we can see that the logs do not show any calls to the `FlowerClient.get_parameters` method.


---



##### 3.5.1.2 Starting with a customised strategy

Flower supports several federated learning strategies, such as `FedAvg` and `FedAdagrad`. The strategy determines the algorithm used for federated learning, making it easy to experiment with alternatives.

Let's try to use a different strategy by:

- Replace the strategy in server_fn with the one we want.
- Pass the updated strategy to the simulation.


###### 3.5.1.2.1 Replace the strategy

In [ ]:
def server_fn(context: Context) -> ServerAppComponents:
    # Create FedAdagrad strategy
    strategy = FedAdagrad(
        fraction_fit=0.3,
        fraction_evaluate=0.3,
        min_fit_clients=3,
        min_evaluate_clients=3,
        min_available_clients=NUM_PARTITIONS,
        initial_parameters=ndarrays_to_parameters(params),
    )
    # Configure the server for 20 rounds of training
    config = ServerConfig(num_rounds=20)
    return ServerAppComponents(strategy=strategy, config=config)

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

###### 3.5.1.2.2 Pass the strategy to the simulation

In [ ]:
# Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)

INFO :      Starting Flower ServerApp, config: num_rounds=20, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 5)
(pid=3292) 2025-04-11 09:03:33.508578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=3292) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=3292) E0000 00:00:1744362213.550322    3292 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=3292) E0000 00:00:1744362213.562654    3292 cuda_blas.cc:1418] Unable to register cuBLAS factory: Att

(ClientAppActor pid=3292) [Client 2] fit, config: {}


(ClientAppActor pid=3292) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3292)   return bound(*args, **kwds)


(ClientAppActor pid=3292) [Client 3] fit, config: {}


(ClientAppActor pid=3292) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3292)   return bound(*args, **kwds)
INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)


...

(ClientAppActor pid=3292) [Client 3] fit, config: {}


(ClientAppActor pid=3292) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3292)   return bound(*args, **kwds)
(ClientAppActor pid=3292) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3292)   return bound(*args, **kwds)


(ClientAppActor pid=3292) [Client 0] evaluate, config: {}
(ClientAppActor pid=3292) [Client 2] evaluate, config: {}


(ClientAppActor pid=3292) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3292)   return bound(*args, **kwds)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 20 round(s) in 43.60s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.016687533747151314
INFO :      		round 2: 0.01653114890256347
INFO :      		round 3: 0.015908180036874303
INFO :      		round 4: 0.01689346336046825
INFO :      		round 5: 0.01670657842557899
INFO :      		round 6: 0.01559558008727863
INFO :      		round 7: 0.01666996918804397
INFO :      		round 8: 0.014996520698499875
INFO :      		round 9: 0.01556499115656811
INFO :      		round 10: 0.014720711586282926
INFO :      		round 11: 0.015908916618848298
INFO :      		round 12: 0.015013495146981777

(ClientAppActor pid=3292) [Client 3] evaluate, config: {}


#### 3.6 Server-side parameter **evaluation**

We've seen how federated evaluation works on the client side (i.e., by implementing the `evaluate` method in `FlowerClient`). Now let's see how we can evaluate aggregated model parameters on the server-side.






---

***Note***

Flower can evaluate the aggregated model on the server-side or on the client-side:

- **Centralised Evaluation** (or *server-side evaluation*):
  - Uses a fixed dataset on the server.
  - Provides consistent and stable results across rounds as the dataset doesn't change.
  - Ideal for scenarios where a curated dataset (e.g. anonymised historical records) is available.

- **Federated Evaluation** (or *client-side evaluation*):
  - Evaluates the global model on client data.
  - More realistic but introduces variability, as client datasets can change over time.


---



**Updated use case scenario**

LifeMed maintains a curated, anonymised dataset on a central server for stable and consistent evaluation of the aggregated model. This **server-side evaluation** ensures that model improvements are assessed independently of client data variability, making it ideal for tracking performance across training rounds. It also provides a reliable benchmark even when some healthcare providers experience temporary data availability issues.  

By using server-side evaluation, LifeMed can monitor generalisability without relying on individual provider data, making it suitable for scenarios where a fixed, representative dataset is available for validation.

###### 3.6.1 Define and evaluate function

We start by defining an `evaluate_fn` that evaluates the global model on the server-side dataset.

In [ ]:
# The `evaluate` function will be called by Flower after every round
def evaluate(
    server_round: int,
    parameters: NDArrays,
    config: Dict[str, Scalar],
) -> Optional[Tuple[float, Dict[str, Scalar]]]:

    _, _, testloader = load_data(0, NUM_PARTITIONS)
    net = StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]).to(DEVICE)
    set_parameters(net, parameters)  # Update model with the latest parameters
    loss, accuracy = test(net, testloader)
    print(f"Server-side evaluation loss {loss} / accuracy {accuracy}")
    return loss, {"accuracy": accuracy}

###### 3.6.2 Pass it to the strategy

In [ ]:
def server_fn(context: Context) -> ServerAppComponents:
    # Create the FedAvg strategy
    strategy = FedAvg(
        fraction_fit=0.3,
        fraction_evaluate=0.3,
        min_fit_clients=3,
        min_evaluate_clients=3,
        min_available_clients=NUM_PARTITIONS,
        initial_parameters=ndarrays_to_parameters(params),
        evaluate_fn=evaluate,  # Pass the evaluation function
    )
    # Configure the server for 20 rounds of training
    config = ServerConfig(num_rounds=20)
    return ServerAppComponents(strategy=strategy, config=config)

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

###### 3.6.3 Run the simulation

Run the simulation. The server will evaluate the model after each round.


In [ ]:
# Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)

INFO :      Starting Flower ServerApp, config: num_rounds=20, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
INFO :      initial parameters (loss, other metrics): 0.021707322034570906, {'accuracy': 0.4409722222222222}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 5)


Server-side evaluation loss 0.021707322034570906 / accuracy 0.4409722222222222


(pid=3851) 2025-04-11 09:04:20.968807: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=3851) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=3851) E0000 00:00:1744362260.999928    3851 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=3851) E0000 00:00:1744362261.007456    3851 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(ClientAppActor pid=3851) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3851)   return bound(*args, **kwds)


(ClientAppActor pid=3851) [Client 1] fit, config: {}


(ClientAppActor pid=3851) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3851)   return bound(*args, **kwds)


...

(ClientAppActor pid=3851) [Client 3] fit, config: {}
Server-side evaluation loss 0.012157696092294322 / accuracy 0.8472222222222222


(ClientAppActor pid=3851) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3851)   return bound(*args, **kwds)
(ClientAppActor pid=3851) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3851)   return bound(*args, **kwds)
(ClientAppActor pid=3851) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=3851)   return bound(*args, **kwds)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 20 round(s) in 4

(ClientAppActor pid=3851) [Client 2] evaluate, config: {}
(ClientAppActor pid=3851) [Client 3] evaluate, config: {}


INFO :      	History (metrics, centralized):
INFO :      	{'accuracy': [(0, 0.4409722222222222),
INFO :      	              (1, 0.7708333333333334),
INFO :      	              (2, 0.8055555555555556),
INFO :      	              (3, 0.8055555555555556),
INFO :      	              (4, 0.8506944444444444),
INFO :      	              (5, 0.8368055555555556),
INFO :      	              (6, 0.8402777777777778),
INFO :      	              (7, 0.8541666666666666),
INFO :      	              (8, 0.8368055555555556),
INFO :      	              (9, 0.8576388888888888),
INFO :      	              (10, 0.84375),
INFO :      	              (11, 0.8472222222222222),
INFO :      	              (12, 0.8680555555555556),
INFO :      	              (13, 0.8055555555555556),
INFO :      	              (14, 0.8472222222222222),
INFO :      	              (15, 0.8506944444444444),
INFO :      	              (16, 0.8402777777777778),
INFO :      	              (17, 0.8923611111111112),
INFO :      	         

(ClientAppActor pid=3851) [Client 4] evaluate, config: {}





---
***Note:***
This approach ensures consistent evaluation even if some clients are unavailable during training rounds.


---



#### 3.7 Sending/receiving arbitrary values to/from clients

Sometimes we need to send configuration values (e.g. `local_epochs` or `learning_rate`) from the server to the clients to customise their behaviour.

This is particularly useful for fine-tuning training or to ensure fair participation between clients with different dataset sizes.

Flower provides an easy way to do this using a configuration dictionary.

For example:
- The server sends values such as `local_epochs` and `server_round` to the clients.
- Clients adjust their training accordingly (e.g. run a different number of epochs).
- Clients can send metrics such as dataset size or training loss back to the server, allowing dynamic weighting during aggregation.

###### 3.7.1 Redefine the client

In [ ]:
class FlowerClient(NumPyClient):
    def __init__(self, pid, net, trainloader, valloader):
        self.pid = pid  # partition ID of a client
        self.net = net
        self.trainloader = trainloader
        self.valloader = valloader

    def get_parameters(self, config):
        print(f"[Client {self.pid}] get_parameters")
        return get_parameters(self.net)

    def fit(self, parameters, config):
        # Read values from config
        server_round = config["server_round"]
        local_epochs = config["local_epochs"]

        # Use values provided by the config
        print(f"[Client {self.pid}, round {server_round}] fit, config: {config}")
        set_parameters(self.net, parameters)
        train(self.net, self.trainloader, epochs=local_epochs)
        return get_parameters(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        print(f"[Client {self.pid}] evaluate, config: {config}")
        set_parameters(self.net, parameters)
        loss, accuracy = test(self.net, self.valloader)
        return float(loss), len(self.valloader), {"accuracy": float(accuracy)}

###### 3.7.2 Define the client app

In [ ]:
def client_fn(context: Context) -> Client:
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    trainloader, valloader, _ = load_data(partition_id, num_partitions)

    net = StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]).to(DEVICE)
    return FlowerClient(partition_id, net, trainloader, valloader).to_client()


# Create the ClientApp
client = ClientApp(client_fn=client_fn)

###### 3.7.3 Define a configuration dictionary

So how can we  send this config dictionary from server to clients? The built-in Flower Strategies provide way to do this, and it works similarly to the way server-side evaluation works:

- Define a function to create the configuration dictionary.
- Pass this function to the strategy.
- Flower will call this function every round and send the generated values to the clients.


In [ ]:
def fit_config(server_round: int):
    """Return training configuration dict for each round.

    Perform two rounds of training with one local epoch, increase to two local
    epochs afterwards.
    """
    config = {
        "server_round": server_round,  # The current round of federated learning
        "local_epochs": 1 if server_round < 2 else 2,
    }
    return config

###### 3.7.4 Pass it to the strategy

In [ ]:
def server_fn(context: Context) -> ServerAppComponents:
    # Create FedAvg strategy
    strategy = FedAvg(
        fraction_fit=0.3,
        fraction_evaluate=0.3,
        min_fit_clients=3,
        min_evaluate_clients=3,
        min_available_clients=NUM_PARTITIONS,
        initial_parameters=ndarrays_to_parameters(params),
        evaluate_fn=evaluate,
        on_fit_config_fn=fit_config,  # Pass the fit_config function
    )
    config = ServerConfig(num_rounds=20)
    return ServerAppComponents(strategy=strategy, config=config)

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

###### 3.7.5 Run the simulation

In [ ]:
# Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)

INFO :      Starting Flower ServerApp, config: num_rounds=20, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
INFO :      initial parameters (loss, other metrics): 0.02167485219736894, {'accuracy': 0.4722222222222222}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 5)


Server-side evaluation loss 0.02167485219736894 / accuracy 0.4722222222222222


(pid=4410) 2025-04-11 09:05:06.761228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=4410) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=4410) E0000 00:00:1744362306.789859    4410 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=4410) E0000 00:00:1744362306.798050    4410 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(ClientAppActor pid=4410) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=4410)   return bound(*args, **kwds)


(ClientAppActor pid=4410) [Client 2, round 1] fit, config: {'server_round': 1, 'local_epochs': 1}


(ClientAppActor pid=4410) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=4410)   return bound(*args, **kwds)


...

(ClientAppActor pid=4410) [Client 3, round 20] fit, config: {'server_round': 20, 'local_epochs': 2}


INFO :      fit progress: (20, 0.0111327621464928, {'accuracy': 0.8402777777777778}, 49.35679417199992)
INFO :      configure_evaluate: strategy sampled 3 clients (out of 5)


Server-side evaluation loss 0.0111327621464928 / accuracy 0.8402777777777778
(ClientAppActor pid=4410) [Client 2] evaluate, config: {}


(ClientAppActor pid=4410) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=4410)   return bound(*args, **kwds)
(ClientAppActor pid=4410) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=4410)   return bound(*args, **kwds)
(ClientAppActor pid=4410) /usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
(ClientAppActor pid=4410)   return bound(*args, **kwds)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 20 round(s) in 4

(ClientAppActor pid=4410) [Client 3] evaluate, config: {}
(ClientAppActor pid=4410) [Client 4] evaluate, config: {}


## 4. Challenge: LifeMed Expansion

LifeMed has expanded to 10 hospitals, each with its own data set. To ensure better model performance and resource allocation, they decided to:

- Use `FedAdam` as their federated learning strategy, with server-side parameter initialisation.
- Perform federated evaluation (client-side evaluation).
- Train the model for 10 federated rounds, with each client running 10 local epochs per round.
- Dynamically adjust the learning rate:
  - Use a learning rate of 1.5 for the first 5 rounds (server_round < 5).
  - Use a learning rate of 3.0 for round 5 and beyond (server_round >= 5).


  The solutions can be found at the end of the tutorial.


**Steps to complete the challenge**

You need to implement this configuration using Flower:

<!--1. Set up the FedAdam strategy with server-side parameter initialisation and the given learning rate schedule.
2. Make sure that federated evaluation is enabled.
3. Simulate training with 10 clients over 10 rounds of federated learning.
-->

1. Modify the `train` function.
2. Define a `FlowerClient`.
3. Define the `NUM_PARTITIONS = 10`.
4. Get the initialisation parameters.
5. Define the new config with the different learning rates.
6. Define the `accuracy` function.
7. Create the `FlowerServer` with the `FedAdam` strategy.
8. Run the simulation.


***Hints:***
- Use fit_config_fn to dynamically adjust the learning rate based on server_round.
- Use the FedAdam strategy when configuring the server.

**Instructions to submit the code**

1. Write your code on the code block below and test it out.
2. Save it as txt file.
3. Submit it on the [Distributed Learning forms](https://forms.gle/aqfHMdSSTQv2dJwW7).

In [ ]:
# First Write and test your code here


## 5. Conclusions

In this tutorial, we have shown how to apply **distributed learning** using the Flower framework to train privacy-preserving machine learning models. By using **distributed learning**, institutions can collaboratively improve model accuracy without exposing sensitive patient data.  

We compared **centralised vs. federated training**, highlighting the trade-offs between privacy, model performance and communication overhead. In addition, we explored **server-side evaluation** to provide a stable performance benchmark.  

With further optimisations such as custom aggregation strategies, differential privacy, and secure multi-party computation, federated learning can be enhanced for even greater security and efficiency.

## Solutions

### Solution for LifeMed expansion

A possible solution for 50%-70% accuracy and loss < 0.1 would be:

1. Modify the `train` function.
2. Define a `FlowerClient`.
3. Define the `NUM_PARTITIONS = 10`.
4. Get the initialisation parameters.
5. Define the new config with the different learning rates.
6. Define the `accuracy` function.
7. Create the `FlowerServer` with the `FedAdam` strategy.
8. Run the simulation.

#### 1. Modify the `train` function

In [ ]:
# 1. We need to change the train function to support different learning rates, the
# rest of the function remains the same
def train(model, train_loader, epochs: int, learning_rate: float, verbose=False):
    """Train the model on the training set."""
    criterion = nn.BCEWithLogitsLoss()

    ## We change this line to support different learning rates
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    model.to(DEVICE)
    model.train()

    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE).float()
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()  # Squeeze to match target shape
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            # Metrics
            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities > 0.5).float()
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader.dataset)
        epoch_acc = correct / total
        if verbose:
            print(f"Epoch {epoch+1}: Train Loss {epoch_loss:.4f}, Accuracy {epoch_acc:.4f}")

#### 2. Define a `FlowerClient`

In [ ]:
# 2. We define a FlowerClient
# This block of code is the same as in 3.8
class FlowerClient(NumPyClient):
    def __init__(self, pid, net, trainloader, valloader):
        self.pid = pid  # partition ID of a client
        self.net = net
        self.trainloader = trainloader
        self.valloader = valloader

    def get_parameters(self, config):
        print(f"[Client {self.pid}] get_parameters")
        return get_parameters(self.net)

    def fit(self, parameters, config):
        # Read values from config
        server_round = config["server_round"]
        local_lr = config["local_lr"]

        # Use values provided by the config
        print(f"[Client {self.pid}, round {server_round}] fit, config: {config}")
        set_parameters(self.net, parameters)
        train(self.net, self.trainloader, epochs=10, learning_rate=local_lr)
        return get_parameters(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        print(f"[Client {self.pid}] evaluate, config: {config}")
        set_parameters(self.net, parameters)
        loss, accuracy = test(self.net, self.valloader)
        return float(loss), len(self.valloader), {"accuracy": float(accuracy)}


def client_fn(context: Context) -> Client:
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    trainloader, valloader, _ = load_data(partition_id, num_partitions)

    net = StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]).to(DEVICE)

    return FlowerClient(partition_id, net, trainloader, valloader).to_client()


# Create the ClientApp
client = ClientApp(client_fn=client_fn)

#### 3. Define the number of partitions

In [ ]:
# 3. Define number of partitions
# Constants
NUM_PARTITIONS = 10 # Number of providers

#### 4. Get the parameters

In [ ]:
# 4. Create an instance of the model and get the parameters
params = get_parameters(StrokePredictionModel(input_dim=trainloader.dataset[0][0].shape[0]))

#### 5. Define the new config with the different learning rates

In [ ]:
# 5. Define the new config with the different learning rates
def fit_config(server_round: int):
    """Return training configuration dict for each round.

    Perform two rounds of training with one local epoch, increase to two local
    epochs afterwards.
    """
    config = {
        "server_round": server_round,  # The current round of federated learning
        "local_lr": 1.5 if server_round < 5 else 3,
    }
    return config

#### 6. Define the `accuracy`

In [ ]:
# 6. Define the accuracy function
# This is the same function as in 3.5
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metric (weighted average)
    return {"accuracy": sum(accuracies) / sum(examples)}

#### 7. Create the `FlowerServer` with the `FedAdam` strategy

In [ ]:
from flwr.server.strategy import FedAdam

# 7. Create the FlowerServer with the FedAdam strategy
def server_fn(context: Context) -> ServerAppComponents:
    # Create FedAdam strategy
    strategy = FedAdam(
        fraction_fit=1.0,  # Sample 100% of available clients for training
        fraction_evaluate=0.5,  # Sample 50% of available clients for evaluation
        min_fit_clients=5,  # Never sample less than 5 clients for training
        min_evaluate_clients=5,  # Never sample less than 5 clients for evaluation
        min_available_clients=NUM_PARTITIONS,  # Wait until all 10 clients are available
        evaluate_metrics_aggregation_fn=weighted_average,  # <-- pass the metric aggregation function
        initial_parameters=ndarrays_to_parameters(params),
        on_fit_config_fn=fit_config,  # Pass the fit_config function
    )
    # Configure the server for 10 rounds of training
    config = ServerConfig(num_rounds=10)
    return ServerAppComponents(strategy=strategy, config=config)

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

#### 8. Run the simulation

In [ ]:
# 8. Run simulation
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_PARTITIONS,
    backend_config=backend_config,
)